# 🛰️ Analisis Komparasi Menyeluruh Curah Hujan Satelit & Reanalisis Harian (2004 – Juli 2026)
### 📍 Wilayah Studi: Kabupaten Kebumen, Jawa Tengah | Matriks Evaluasi Antar-Variabel (*All-to-All Matrix*) & Poros Benchmark Kontinu `CHIRPS_RNL`

---
### 📌 Ringkasan Eksekutif (*Executive Summary*)
Notebook ini menyajikan evaluasi hidrometeorologi komprehensif dari **8 Produk Presipitasi Harian** (Satelit dan Reanalisis Atmosfer) selama **8.248 hari kalender kontinu (1 Januari 2004 hingga 31 Juli 2026)** di Kabupaten Kebumen, Jawa Tengah.

#### 🎯 8 Variabel Presipitasi yang Dianalisis:
1. **`CHIRPS_RNL`**: CHIRPS Reanalysis (*Ground-Truth Benchmark*, kelengkapan 100% tanpa celah).
2. **`CHIRPS_SAT`**: CHIRPS Satellite-Only (Estimasi murni satelit).
3. **`CHIRPS_FNL`**: CHIRPS Final (Terkoreksi stasiun darat global).
4. **`GSMaP`**: JAXA Global Satellite Mapping of Precipitation.
5. **`IMERG`**: NASA GPM Integrated Multi-satellitE Retrievals (Final V06/V07).
6. **`PERSIANN`**: PERSIANN-CDR (Climate Data Record, NOAA / UC Irvine).
7. **`ERA5`**: ECMWF Reanalysis Generasi ke-5 Global (~31 km).
8. **`ERA5_LAND`**: ECMWF Reanalysis Daratan Resolusi Tinggi (~9 km).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

print("Library dan environment berhasil dimuat.")


## 📂 1. Pemuatan Dataset & Audit Integritas Data (2004 – Juli 2026)
Memuat berkas `Data_Curah_Hujan_Kebumen.csv`, memfilter rentang waktu **1 Januari 2004 s.d. 31 Juli 2026 (8.248 hari)**, serta mengisolasi 8 variabel satelit dan reanalisis.


In [ ]:
data_path = r'../Data_Satelit/Data_Curah_Hujan_Kebumen.csv'
if not os.path.exists(data_path):
    data_path = r'd:/Github/Projek_Rainfall/Google_Earth_Engine/Data_Satelit/Data_Curah_Hujan_Kebumen.csv'

df_raw = pd.read_csv(data_path)
df_raw['Date'] = pd.to_datetime(df_raw['datetime_utc'] if 'datetime_utc' in df_raw.columns else df_raw['Date'])

# Filter hingga 31 Juli 2026
df_filtered = df_raw[(df_raw['Date'] >= '2004-01-01') & (df_raw['Date'] <= '2026-07-31')].sort_values('Date').reset_index(drop=True)

var_cols = ['CHIRPS_RNL', 'CHIRPS_SAT', 'CHIRPS_FNL', 'GSMaP', 'IMERG', 'PERSIANN', 'ERA5', 'ERA5_LAND']
df = df_filtered.set_index('Date')[var_cols].copy()

# Koreksi nilai negatif mikro ERA5
for c in ['ERA5', 'ERA5_LAND']:
    df[c] = df[c].clip(lower=0.0)

print(f"Total Hari Pengamatan (2004 s.d. Juli 2026): {len(df):,} hari")
display(df.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])


## 📊 2. Matriks Evaluasi Inter-Model All-to-All ($8 	imes 8 = 64$ Pasangan)
Evaluasi korelasi Pearson ($r$), Spearman ($ho$), RMSE, MAE, Kling-Gupta Efficiency (KGE), dan Percent Bias (PBIAS).


In [ ]:
pairwise_csv = r'Hasil_Analisis_Harian_2004_2026/ringkasan_evaluasi_all_pairs.csv'
if os.path.exists(pairwise_csv):
    df_eval = pd.read_csv(pairwise_csv)
    display(df_eval.sort_values(by='Spearman_rho', ascending=False).head(10))


## 🖼️ 3. Visualisasi Hasil Analisis
Seluruh 13 grafik visualisasi hasil analisis resolusi tinggi disimpan di folder `Hasil_Analisis_Harian_2004_2026/`.


In [ ]:
from IPython.display import Image, display

plots = [
    '01_scatterplot_matrix_curah_hujan.png',
    '02_heatmap_korelasi_pearson_spearman.png',
    '02b_matriks_evaluasi_error_all_to_all.png',
    '03b_scatter_hexbin_vs_chirps_rnl.png',
    '04b_bar_akurasi_vs_chirps_rnl.png',
    '05b_kategorikal_skill_vs_chirps_rnl.png',
    '06_tren_akumulasi_hujan_tahunan_2004_2025.png',
    '07_timeseries_30day_moving_average.png',
    '08_kurva_massa_ganda_double_mass_curve.png',
    '09_klimatologi_bulanan_barchart.png',
    '10_boxplot_variabilitas_bulanan.png',
    '11_distribusi_pdf_cdf_intensitas_hujan.png',
    '12_tren_anomali_curah_hujan_bulanan.png'
]

for p in plots:
    fp = os.path.join('Hasil_Analisis_Harian_2004_2026', p)
    if os.path.exists(fp):
        print(f'=== {p} ===')
        display(Image(fp))


## 🎯 4. Kesimpulan & Rekomendasi Terapan Hidrologi
1. **Konsensus Satelit Tertinggi**: `CHIRPS_SAT` dan `NASA GPM IMERG` ($r = 0.810, ho = 0.875, 	ext{KGE} = 0.686$).
2. **Konsistensi Reanalisis Atmosfer**: `CHIRPS_RNL` vs `ERA5_LAND` ($ho = 0.824, 	ext{RMSE} = 8.38	ext{ mm/hari}$).
3. **Rekomendasi Pemodelan DAS**: Gunakan `CHIRPS_RNL` untuk pemodelan kontinu tanpa gap (2004–2026), dan `GPM IMERG` untuk deteksi hujan lebat dan mitigasi banjir.
